# Download NCBI data

As the NCBI data contains TIGRFAM, GO and COG data, download everything at once.

In [ ]:
import gzip, requests, re, time, sys
from Bio import Entrez, SeqIO
import pandas as pd

In [2]:
Entrez.email = "jonas.hagenberg@helmholtz-munich.de" 

In [3]:
with open("../data/example_wp_ids.txt", "r") as f:
        ids = [l.strip() for l in f if l.strip()]

In [ ]:
PAT_TIGR = re.compile(r"TIGR\d{5}|NF\d{6}")
url = "https://ftp.ncbi.nih.gov/pub/mmdb/cdd/cddid.tbl.gz"

with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open("../data/cddid.tbl.gz", "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):  # 1 MB at a time
            f.write(chunk)
cdd = {}
with gzip.open("../data/cddid.tbl.gz", "rt") as fh:
    for line in fh:
        f = line.rstrip("\n").split("\t")
        if len(f) > 1 and PAT_TIGR.fullmatch(f[1]):
            cdd[f[0]] = f[1]

In [12]:
PAT_COG = re.compile(r"COG\d{4}")
PAT_GO = re.compile(r"GO:.*?(?=\s*\[Evidence|$)")
found_tigr = {i: set() for i in ids}
found_go = {i: set() for i in ids}
found_cog = {i: set() for i in ids}
for s in range(0, len(ids), 100):
    batch = ids[s:s+100]
    h = Entrez.efetch(db="protein", id=",".join(batch), rettype="gp", retmode="text")
    for rec in SeqIO.parse(h, "genbank"):
        for f in rec.features:
            if f.type == "Region":
                # TIGRFAM
                q = f.qualifiers
                text = " ".join(q.get("region_name", []) + q.get("note", []))
                hits = set(PAT_TIGR.findall(text))
                for x in q.get("db_xref", []):
                    if x.startswith("CDD:") and x[4:] in cdd:
                        hits.add(cdd[x[4:]])
                found_tigr[rec.id].update(hits)
                # COG
                text_cog = " ".join(q.get("note", []))
                hits_cog = set(PAT_COG.findall(text_cog))
                found_cog[rec.id].update(hits_cog)
            elif f.type == "Protein":
                # GO Process
                q = f.qualifiers
                go_data = " ".join(q.get("GO_process", []))
                hits_go = set(PAT_GO.findall(go_data))
                found_go[rec.id].update(hits_go)


    h.close()
    time.sleep(0.4)

In [13]:
found_tigr

{'WP_000942300.1': set(),
 'WP_000602071.1': set(),
 'WP_000524836.1': set(),
 'WP_001229077.1': set(),
 'WP_001172340.1': set(),
 'WP_000840574.1': set(),
 'WP_000148605.1': set(),
 'WP_001151902.1': set(),
 'WP_000069282.1': set(),
 'WP_000745871.1': {'NF033845'},
 'WP_011447028.1': set(),
 'WP_000409167.1': set(),
 'WP_000019687.1': set(),
 'WP_001006454.1': {'TIGR01979'},
 'WP_000290485.1': set(),
 'WP_000280803.1': set(),
 'WP_000884334.1': set(),
 'WP_000250824.1': {'TIGR01825'},
 'WP_000202192.1': set(),
 'WP_000668820.1': set(),
 'WP_000076036.1': set(),
 'WP_000866742.1': set(),
 'WP_001281145.1': set(),
 'WP_000723301.1': set(),
 'WP_001036663.1': set(),
 'WP_000808845.1': {'TIGR01327'},
 'WP_000120494.1': set(),
 'WP_001140234.1': {'TIGR00719'},
 'WP_000460421.1': set(),
 'WP_000708692.1': set(),
 'WP_000383180.1': set(),
 'WP_000023316.1': set(),
 'WP_000866415.1': set(),
 'WP_001292247.1': set(),
 'WP_000276366.1': {'TIGR01998'},
 'WP_001124985.1': set(),
 'WP_000579275.1'

In [14]:
found_cog

{'WP_000942300.1': {'COG0488'},
 'WP_000602071.1': {'COG0488'},
 'WP_000524836.1': {'COG4152'},
 'WP_001229077.1': set(),
 'WP_001172340.1': {'COG0365'},
 'WP_000840574.1': {'COG0827'},
 'WP_000148605.1': set(),
 'WP_001151902.1': set(),
 'WP_000069282.1': set(),
 'WP_000745871.1': set(),
 'WP_011447028.1': {'COG1104'},
 'WP_000409167.1': {'COG1104'},
 'WP_000019687.1': set(),
 'WP_001006454.1': set(),
 'WP_000290485.1': set(),
 'WP_000280803.1': {'COG0095'},
 'WP_000884334.1': set(),
 'WP_000250824.1': set(),
 'WP_000202192.1': set(),
 'WP_000668820.1': set(),
 'WP_000076036.1': set(),
 'WP_000866742.1': set(),
 'WP_001281145.1': {'COG0787'},
 'WP_000723301.1': set(),
 'WP_001036663.1': {'COG0626'},
 'WP_000808845.1': set(),
 'WP_000120494.1': set(),
 'WP_001140234.1': set(),
 'WP_000460421.1': set(),
 'WP_000708692.1': set(),
 'WP_000383180.1': {'COG2049'},
 'WP_000023316.1': {'COG1984'},
 'WP_000866415.1': set(),
 'WP_001292247.1': set(),
 'WP_000276366.1': set(),
 'WP_001124985.1':

In [15]:
found_go

{'WP_000942300.1': set(),
 'WP_000602071.1': set(),
 'WP_000524836.1': set(),
 'WP_001229077.1': set(),
 'WP_001172340.1': set(),
 'WP_000840574.1': {'GO:0032259 - methylation'},
 'WP_000148605.1': {'GO:0006457 - protein folding'},
 'WP_001151902.1': set(),
 'WP_000069282.1': set(),
 'WP_000745871.1': {'GO:0007155 - cell adhesion'},
 'WP_011447028.1': set(),
 'WP_000409167.1': set(),
 'WP_000019687.1': {'GO:0006546 - glycine catabolic process'},
 'WP_001006454.1': {'GO:0006534 - cysteine metabolic process'},
 'WP_000290485.1': {'GO:0019464 - glycine decarboxylation via glycine cleavage system'},
 'WP_000280803.1': {'GO:0009249 - protein lipoylation'},
 'WP_000884334.1': {'GO:0006434 - seryl-tRNA aminoacylation'},
 'WP_000250824.1': {'GO:0006567 - L-threonine catabolic process'},
 'WP_000202192.1': {'GO:0006546 - glycine catabolic process'},
 'WP_000668820.1': {'GO:0009249 - protein lipoylation'},
 'WP_000076036.1': {'GO:0009081 - branched-chain amino acid metabolic process'},
 'WP_0008

In [16]:
found_results_tigr = {key: val for key, val in found_tigr.items() if val}
found_results_tigr = (
    pd.Series(found_results_tigr, name="tigr_acc")
    .explode()
    .rename_axis("wp_id")
    .reset_index()
)

In [22]:
found_results_cog = {key: val for key, val in found_cog.items() if val}
found_results_cog = (
    pd.Series(found_results_cog, name="cog_acc")
    .explode()
    .rename_axis("wp_id")
    .reset_index()
)

In [21]:
found_results_go = {key: val for key, val in found_go.items() if val}
found_results_go = (
    pd.Series(found_results_go, name="go_acc")
    .explode()
    .rename_axis("wp_id")
    .reset_index()
)

In [23]:
found_results_tigr.to_csv("../data/example_results_tigr.tsv", sep="\t", index=False)
found_results_cog.to_csv("../data/example_results_cog.tsv", sep="\t", index=False)
found_results_go.to_csv("../data/example_results_go.tsv", sep="\t", index=False)